In [2]:
import sqlite3

import sqlite3
import os

# This works regardless of where the notebook is running from
BASE_DIR = os.path.dirname(os.path.dirname(os.path.abspath("__file__")))
DB_PATH = os.path.join(BASE_DIR, "data", "tradeflow.db")

conn = sqlite3.connect(DB_PATH)
print(f"Connected to: {DB_PATH}")

markets = [
    (1, "Mile 12 Market",  "Lagos"),
    (1, "Oyingbo Market",  "Lagos"),
    (2, "Bodija Market",   "Ibadan"),
    (2, "Oje Market",      "Ibadan"),
    (3, "Sagamu Market",   "Sagamu"),
    (4, "Oja-Oba Market",  "Ilorin"),
    (5, "Ganaja Market",   "Lokoja"),
    (6, "Wuse Market",     "Abuja"),
    (6, "Karu Market",     "Abuja"),
    (7, "Lafia Market",    "Lafia"),
    (8, "Minna Market",    "Minna"),
]

conn.executemany(
    "INSERT INTO markets (state_id, name, city) VALUES (?, ?, ?)",
    markets
)
conn.commit()
conn.close()
print("✓ Markets inserted")

Connected to: C:\Users\USER\Projects\TradeFlow\data\tradeflow.db
✓ Markets inserted


In [4]:
import sqlite3

import sqlite3
import os

# This works regardless of where the notebook is running from
BASE_DIR = os.path.dirname(os.path.dirname(os.path.abspath("__file__")))
DB_PATH = os.path.join(BASE_DIR, "data", "tradeflow.db")

conn = sqlite3.connect(DB_PATH)
print(f"Connected to: {DB_PATH}")

corridors = [
    (8, 6, 230,  3.5, "Good"),   # Niger → Abuja
    (7, 6, 100,  1.5, "Good"),   # Nasarawa → Abuja
    (5, 6, 180,  3.0, "Fair"),   # Kogi → Abuja
    (6, 4, 300,  5.0, "Fair"),   # Abuja → Kwara
    (4, 2, 130,  2.5, "Fair"),   # Kwara → Oyo
    (5, 2, 280,  5.0, "Poor"),   # Kogi → Oyo
    (2, 1, 120,  2.0, "Good"),   # Oyo → Lagos
    (2, 3,  80,  1.5, "Good"),   # Oyo → Ogun
    (3, 1,  60,  1.0, "Good"),   # Ogun → Lagos
    (8, 2, 580,  9.0, "Fair"),   # Niger → Oyo
    (7, 2, 480,  8.0, "Fair"),   # Nasarawa → Oyo
    (6, 1, 520,  8.5, "Good"),   # Abuja → Lagos
]

conn.executemany("""
    INSERT INTO corridors
    (origin_state_id, dest_state_id, distance_km, avg_travel_hours, road_quality)
    VALUES (?, ?, ?, ?, ?)
""", corridors)
conn.commit()
conn.close()
print("✓ Corridors inserted")

Connected to: C:\Users\USER\Projects\TradeFlow\data\tradeflow.db
✓ Corridors inserted


In [3]:
import sqlite3
import numpy as np
from datetime import date, timedelta

import sqlite3
import os

# This works regardless of where the notebook is running from
BASE_DIR = os.path.dirname(os.path.dirname(os.path.abspath("__file__")))
DB_PATH = os.path.join(BASE_DIR, "data", "tradeflow.db")

conn = sqlite3.connect(DB_PATH)
print(f"Connected to: {DB_PATH}")

np.random.seed(42)

base_prices = {
    1: {6:18000, 7:16000, 8:17000, 4:20000, 5:21000, 2:25000, 1:28000, 3:26000},  # Yam
    2: {6:28000, 7:26000, 8:27000, 4:30000, 5:31000, 2:35000, 1:38000, 3:36000},  # Maize
    3: {6:45000, 7:43000, 8:44000, 4:47000, 5:48000, 2:52000, 1:55000, 3:53000},  # Rice
    4: {6: 3500, 7: 3200, 8: 3300, 4: 3800, 5: 4000, 2: 5500, 1: 6500, 3: 6000},  # Tomato
}

records = []
start_date = date.today() - timedelta(weeks=8)

for week in range(8):
    price_date = start_date + timedelta(weeks=week)
    for commodity_id, state_prices in base_prices.items():
        for state_id, base in state_prices.items():
            noise = np.random.normal(0, base * 0.05)
            price = round(max(base + noise, base * 0.7), 2)
            records.append((
                state_id, commodity_id,
                price, price / 100,
                price_date, False, True
            ))

conn.executemany("""
    INSERT OR IGNORE INTO cleaned_prices
    (state_id, commodity_id, price_per_unit, price_per_kg,
     price_date, is_outlier, is_confirmed)
    VALUES (?, ?, ?, ?, ?, ?, ?)
""", records)

conn.commit()
print(f"✓ {len(records)} dummy price records inserted")
conn.close()

Connected to: C:\Users\USER\Projects\TradeFlow\data\tradeflow.db
✓ 256 dummy price records inserted


C:\Users\USER\AppData\Local\Temp\ipykernel_14316\1512231728.py:39: DeprecationWarning: The default date adapter is deprecated as of Python 3.12; see the sqlite3 documentation for suggested replacement recipes
  conn.executemany("""


In [5]:
import sqlite3
import pandas as pd

import sqlite3
import os

# This works regardless of where the notebook is running from
BASE_DIR = os.path.dirname(os.path.dirname(os.path.abspath("__file__")))
DB_PATH = os.path.join(BASE_DIR, "data", "tradeflow.db")

conn = sqlite3.connect(DB_PATH)
print(f"Connected to: {DB_PATH}")

for table in ["states", "markets", "corridors", "commodities", "cleaned_prices"]:
    count = pd.read_sql(f"SELECT COUNT(*) as n FROM {table}", conn).iloc[0,0]
    print(f"  {table:<20} → {count} rows")

conn.close()

Connected to: C:\Users\USER\Projects\TradeFlow\data\tradeflow.db
  states               → 8 rows
  markets              → 11 rows
  corridors            → 12 rows
  commodities          → 9 rows
  cleaned_prices       → 256 rows


In [8]:
import sqlite3, pandas as pd
import sqlite3
import os

# This works regardless of where the notebook is running from
BASE_DIR = os.path.dirname(os.path.dirname(os.path.abspath("__file__")))
DB_PATH = os.path.join(BASE_DIR, "data", "tradeflow.db")

conn = sqlite3.connect(DB_PATH)
print(f"Connected to: {DB_PATH}")

df = pd.read_sql("SELECT * FROM cleaned_prices LIMIT 20", conn)
df

Connected to: C:\Users\USER\Projects\TradeFlow\data\tradeflow.db


,id,raw_submission_id,state_id,market_id,commodity_id,price_per_unit,price_per_kg,quantity_available,price_date,is_outlier,outlier_reason,is_confirmed,cleaning_notes,cleaned_at
0,1,None,6,None,1,18447.04,184.4704,None,2026-02-15,0,None,1,None,2026-04-12 09:20:56
1,2,None,7,None,1,15889.39,158.8939,None,2026-02-15,0,None,1,None,2026-04-12 09:20:56
2,3,None,8,None,1,17550.54,175.5054,None,2026-02-15,0,None,1,None,2026-04-12 09:20:56
3,4,None,4,None,1,21523.03,215.2303,None,2026-02-15,0,None,1,None,2026-04-12 09:20:56
4,5,None,5,None,1,20754.14,207.5414,None,2026-02-15,0,None,1,None,2026-04-12 09:20:56
5,6,None,2,None,1,24707.33,247.0733,None,2026-02-15,0,None,1,None,2026-04-12 09:20:56
6,7,None,1,None,1,30210.90,302.1090,None,2026-02-15,0,None,1,None,2026-04-12 09:20:56
7,8,None,3,None,1,26997.67,269.9767,None,2026-02-15,0,None,1,None,2026-04-12 09:20:56
8,9,None,6,None,2,27342.74,273.4274,None,2026-02-15,0,None,1,None,2026-04-12 09:20:56
9,10,None,7,None,2,26705.33,267.0533,None,2026-02-15,0,None,1,None,2026-04-12 09:20:56
